# StopTheCap - Colab resource measurement (private project)

Measures **what this Colab account actually provides**. It does **not** train E1 and does not
touch the split manifest, the frozen threshold, or any dataset.

**Privacy:** the project is loaded from a ZIP in *your* Google Drive. Nothing is uploaded to a
public service, no repository is cloned, and no GitHub credentials are requested.

**Recorded:** GPU model and VRAM, CUDA availability, CPU, RAM, disk, torch and transformers
versions, encoder throughput, peak VRAM, and whether a checkpoint directory is writable.

**Not recordable by code:** remaining GPU quota and session timeout. Colab does not expose them;
you fill those two in by hand at the end.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"
print("Drive mounted at", DRIVE_ROOT)

## 2. Configure how to load the project

In [ ]:
# Edit only if your paths differ.
LOAD_METHOD = "zip"                                    # "zip" | "git" | "auto"
ZIP_PATH    = "/content/drive/MyDrive/StopTheCap.zip"  # private ZIP in your Drive
REPO_URL    = ""                                       # only used by "git" mode

REPO_DIR       = "/content/StopTheCap"
REPORT_DIR     = "/content/drive/MyDrive/StopTheCap_reports"
CHECKPOINT_DIR = REPORT_DIR + "/ckpt_probe"

print("method   :", LOAD_METHOD)
print("zip path :", ZIP_PATH)
print("repo dir :", REPO_DIR)

## 3. Load the project (private ZIP from Drive, or Git)

In [ ]:
# Changing the runtime type restarts the kernel, which wipes the config cell. Restore sane
# defaults so this cell works either way instead of dying with NameError.
if "LOAD_METHOD" not in globals():
    LOAD_METHOD = "zip"
    ZIP_PATH = "/content/drive/MyDrive/StopTheCap.zip"
    REPO_DIR = "/content/StopTheCap"
    print("NOTE: the config cell had not run in this session; using default paths.")
    print("      Re-run cell 2 if your ZIP lives somewhere else.")

import os, shutil, sys, zipfile
from pathlib import Path

# The probe needs only these. datasets/ is deliberately optional: measurement does not use data.
REQUIRED = ["ml/__init__.py", "ml/cloud/__init__.py", "ml/cloud/probe.py"]
EXTRACT_TMP = "/content/_stc_extract"   # overridden in tests

def _fail(msg):
    raise RuntimeError(msg)

def load_from_zip(zip_path, dest):
    z = Path(zip_path)
    if not z.is_file():
        _fail(
            "ZIP not found: " + str(z) + "\n"
            "Upload your project ZIP to Google Drive, then set ZIP_PATH in the previous cell.\n"
            "Expected form: /content/drive/MyDrive/<name>.zip"
        )
    dest = Path(dest)
    tmp = Path(EXTRACT_TMP)
    for p in (dest, tmp):
        if p.exists():
            shutil.rmtree(p)
    tmp.mkdir(parents=True)

    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
        print("zip entries:", len(names))
        for n in names[:12]:
            print("   ", n)
        if not names:
            _fail("The ZIP is empty.")
        zf.extractall(tmp)

    entries = [p for p in tmp.iterdir() if p.name != "__MACOSX"]
    # Flatten a single top-level folder so we never end up with StopTheCap/StopTheCap.
    root = entries[0] if len(entries) == 1 and entries[0].is_dir() else tmp
    shutil.move(str(root), str(dest))
    if tmp.exists():
        shutil.rmtree(tmp, ignore_errors=True)
    return dest

def load_from_git(url, dest):
    import subprocess
    if not url:
        _fail("REPO_URL is empty, so git mode cannot run. Use LOAD_METHOD = 'zip'.")
    dest = Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=True)
    return dest

if LOAD_METHOD == "zip":
    REPO = load_from_zip(ZIP_PATH, REPO_DIR)
elif LOAD_METHOD == "git":
    REPO = load_from_git(REPO_URL, REPO_DIR)
elif LOAD_METHOD == "auto":
    REPO = load_from_zip(ZIP_PATH, REPO_DIR) if Path(ZIP_PATH).is_file() else load_from_git(REPO_URL, REPO_DIR)
else:
    _fail("Unknown LOAD_METHOD " + repr(LOAD_METHOD) + ". Use 'zip', 'git' or 'auto'.")

REPO = Path(REPO).resolve()
missing = [f for f in REQUIRED if not (REPO / f).is_file()]
if missing:
    _fail(
        "Project structure looks wrong; missing: " + ", ".join(missing) + "\n"
        "The ZIP must contain the repository root - the folder that holds ml/, datasets/, notebooks/."
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print()
print("project root :", REPO)
print("top level    :", sorted(p.name for p in REPO.iterdir())[:14])
if not (REPO / "datasets/manifests/COMBINED.yaml").is_file():
    print("NOTE: datasets/manifests/COMBINED.yaml absent - split digest will not be recorded "
          "(measurement does not need it).")

## 4. Dependencies

In [ ]:
# torch ships with Colab. The probe imports only transformers lazily for the throughput benchmark,
# so that is the only package added. No scikit-learn, no pydantic, no pyyaml here.
%pip install -q "transformers>=4.40"

import torch, transformers
print("torch       :", torch.__version__)
print("cuda build  :", torch.version.cuda)
print("transformers:", transformers.__version__)
print("cuda present:", torch.cuda.is_available())

## 5. Run the measurement

In [ ]:
# 30 optimiser steps on synthetic tensors, to size a real run.
# This is NOT E1 training: no dataset is read, no checkpoint is trained, nothing is saved but a
# tiny probe file. Peak VRAM is reset immediately before the timed loop.
from pathlib import Path
from ml.cloud.probe import run_probe

REPORT_PATH = REPO / "reports/cloud/colab_measure.json"
res = run_probe(
    out=REPORT_PATH,
    benchmark_model="distilbert-base-cased",
    steps=30,
    checkpoint_dir=Path(CHECKPOINT_DIR),
)
print("wrote", REPORT_PATH)
print("benchmark:", res.get("benchmark"))

## 6. Save the report to Drive

In [ ]:
import shutil
from pathlib import Path

Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)
drive_copy = Path(REPORT_DIR) / "colab_measure.json"
shutil.copy2(REPORT_PATH, drive_copy)
print("persistent copy:", drive_copy)
print("bytes          :", drive_copy.stat().st_size)

## 7. Report

In [ ]:
import json
from pathlib import Path

d = json.loads(Path(REPORT_PATH).read_text(encoding="utf-8"))

print("host       :", d["platform"]["host"])
print("cuda       :", d["gpu"]["cuda_available"])
for dev in d["gpu"].get("devices", []):
    print("GPU        :", dev["name"], "|", dev["total_vram_gb"], "GB")
print("ram        :", d["memory"])
print("disk       :", d["disk"])
print("benchmark  :", d.get("benchmark"))
print("checkpoint :", d.get("checkpoint"))
print()
print(json.dumps(d, indent=2))

## 8. Two values to record by hand

Colab does not expose these to code, so a missing value is left `null` on purpose:

1. **Remaining GPU quota** - open the purchase panel in the Colab UI, then note it in the JSON as
   `quota.measured_gpu_hours_remaining`.
2. **Session timeout** - note it as `quota.measured_session_timeout_minutes`.

## 9. Checkpoint persistence test

1. Note `checkpoint.sha256_prefix` from the report above.
2. Choose **Runtime -> Disconnect and delete runtime**.
3. Reconnect, re-run cells 1, 2, 3 and 7 only.
4. If `probe.txt` in `StopTheCap_reports/ckpt_probe` still exists with the same hash, set
   `checkpoint.survives_disconnect = true`.

The first run only proves the directory is writable; the disconnect test is what proves it is
persistent.

## What this notebook deliberately does NOT do

- No E1 training, no dataset download, no checkpoint of a model.
- No change to split manifests, split digests, class weights or the frozen threshold.
- No publishing, no repository visibility change, no `git push`.
- No upload of source, data or media to any third-party service.